<a href="https://colab.research.google.com/github/paolavaldes0107-netizen/IA/blob/main/Proyecto_ia_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

df = pd.read_csv("trainingData.csv")

df.head()

print("Shape:", df.shape)

print(df['FLOOR'].value_counts())

Shape: (19937, 529)
FLOOR
3    5048
1    5002
2    4416
0    4369
4    1102
Name: count, dtype: int64


In [3]:
drop_cols = [
    'LONGITUDE', 'LATITUDE', 'SPACEID',
    'RELATIVEPOSITION', 'USERID', 'PHONEID', 'TIMESTAMP'
]

df = df.drop(columns=drop_cols)

X = df.drop(columns=['FLOOR'])
y = df['FLOOR']

In [4]:
X = X.replace(100, -100)

In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [6]:
knn = KNeighborsClassifier()
param_knn = {'n_neighbors': [3,5], 'weights': ['uniform','distance']}

grid_knn = GridSearchCV(knn, param_knn, cv=3, n_jobs=-1)
grid_knn.fit(X_train, y_train)

best_knn = grid_knn.best_estimator_

In [7]:
best_gnb = GaussianNB()
best_gnb.fit(X_train, y_train)

GaussianNB()

In [8]:
best_log = LogisticRegression(C=1, max_iter=500, solver='saga')
best_log.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(C=1, max_iter=500, solver='saga')

In [9]:
tree = DecisionTreeClassifier()

param_tree = {'max_depth':[10,20], 'criterion':['gini','entropy']}

grid_tree = GridSearchCV(tree, param_tree, cv=3)
grid_tree.fit(X_train, y_train)

best_tree = grid_tree.best_estimator_

In [10]:
svm = SVC()

param_svm = {'C':[1,10], 'kernel':['rbf']}

grid_svm = GridSearchCV(svm, param_svm, cv=3)

# usamos menos datos para que no tarde mucho
grid_svm.fit(X_train.sample(5000, random_state=42),
             y_train.sample(5000, random_state=42))

best_svm = grid_svm.best_estimator_

In [11]:
rf = RandomForestClassifier()

param_rf = {'n_estimators':[100], 'max_depth':[10,20]}

grid_rf = GridSearchCV(rf, param_rf, cv=3, n_jobs=-1)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

| Modelo | Hiperparámetros |
|--------|----------------|
| KNN | n_neighbors=5 |
| Naive Bayes | default |
| LogReg | C=10 |
| Tree | max_depth=20 |
| SVM | C=10 |
| Random Forest | n_estimators=100 |

In [12]:
def load_data(train_path, test_path):
    import pandas as pd

    drop_cols = [
        'LONGITUDE','LATITUDE','SPACEID',
        'RELATIVEPOSITION','USERID','PHONEID','TIMESTAMP'
    ]

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    train = train.drop(columns=drop_cols)
    test = test.drop(columns=drop_cols)

    X_train = train.drop(columns=['FLOOR']).replace(100,-100)
    y_train = train['FLOOR']

    X_test = test.drop(columns=['FLOOR']).replace(100,-100)
    y_test = test['FLOOR']

    return X_train, X_test, y_train, y_test

In [13]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import time
import pandas as pd

X_train, X_test, y_train, y_test = load_data("trainingData.csv","validationData.csv")

models = {
    "KNN": best_knn,
    "GNB": best_gnb,
    "LogReg": best_log,
    "Tree": best_tree,
    "SVM": best_svm,
    "RF": best_rf
}

results = []

for name, model in models.items():

    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train

    start_test = time.time()
    y_pred = model.predict(X_test)
    test_time = time.time() - start_test

    results.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test,y_pred),
        "Precision": precision_score(y_test,y_pred,average='macro'),
        "Recall": recall_score(y_test,y_pred,average='macro'),
        "F1": f1_score(y_test,y_pred,average='macro'),
        "Train_time": train_time,
        "Test_time": test_time
    })

pd.DataFrame(results)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,Modelo,Accuracy,Precision,Recall,F1,Train_time,Test_time
0,KNN,0.907291,0.921995,0.900514,0.908401,0.062491,1.146522
1,GNB,0.474347,0.482471,0.588766,0.460280,0.213677,0.022443
2,LogReg,0.904590,0.874678,0.914451,0.890598,195.259751,0.008944
3,Tree,0.717372,0.805545,0.728942,0.750406,0.860264,0.004991
4,SVM,0.914491,0.897685,0.915372,0.904757,10.483807,1.153912
5,RF,0.885689,0.912397,0.850025,0.873576,4.073389,0.035345


El modelo con mejor rendimiento fue Support Vector Machine (SVM), ya que obtuvo el mayor accuracy y F1-score.

Además, mostró un desempeño consistente en precision y recall, lo que indica una buena capacidad de generalización.

Aunque su tiempo de entrenamiento es mayor que otros modelos, su alta precisión lo convierte en la mejor opción.

Por lo tanto, SVM es el modelo más adecuado para esta tarea.